# 06 - NetworkX Graph Inspection and Validation

Orthograph provides a two-phase architecture for understanding graph data:

1. **Inspect** -- a `GraphInspector` scans a graph source and produces a `GraphProfile`: a structural summary of node types, relationship types, property completeness, observed types, and cardinality statistics.
2. **Validate** -- the `validate_profile` function compares a `GraphProfile` against a `GraphDataModel` and reports mismatches.

This notebook demonstrates the full workflow using **NetworkX** as the graph backend, with the same filmography domain (Person, Movie, City) used throughout these notebooks.

In [ ]:
from typing import Optional

import networkx as nx

from orthograph import (
    Cardinality,
    GraphDataModel,
    NodeModel,
    RelationshipModel,
)
from orthograph.extensions.networkx import NetworkxInspector, schema_to_networkx
from orthograph.extensions.validation import validate_profile

## Define the model

The familiar filmography model: people act in and direct movies, and live in cities.

In [ ]:
class Person(NodeModel):
    __label__ = "Person"
    __uid_field__ = "name"
    name: str
    age: int
    email: Optional[str] = None


class Movie(NodeModel):
    __label__ = "Movie"
    __uid_field__ = "title"
    title: str
    year: int
    rating: Optional[float] = None


class City(NodeModel):
    __label__ = "City"
    __uid_field__ = "name"
    name: str
    country: str


class ActedIn(RelationshipModel):
    __label__ = "ACTED_IN"
    __source_type__ = Person
    __target_type__ = Movie
    role: str


class Directed(RelationshipModel):
    __label__ = "DIRECTED"
    __source_type__ = Person
    __target_type__ = Movie


class LivesIn(RelationshipModel):
    __label__ = "LIVES_IN"
    __source_type__ = Person
    __target_type__ = City
    __source_cardinality__ = Cardinality.ONE
    __target_cardinality__ = Cardinality.ZERO_OR_MORE


model = GraphDataModel(
    name="Filmography",
    node_types=[Person, Movie, City],
    relationship_types=[ActedIn, Directed, LivesIn],
)

print("Model:", model.name)
print("Nodes:", model.node_labels)
print("Rels: ", model.relationship_labels)

## Build a NetworkX graph with sample data

We populate a `MultiDiGraph` with people, movies, and cities. To demonstrate property completeness analysis, we intentionally:

- Omit the `email` property on all Person nodes (optional, so this is fine)
- Omit the `age` property on one Person node (required -- this will trigger a warning)

In [ ]:
G = nx.MultiDiGraph()

# People -- note p3 is missing the 'age' property
G.add_node("p1", __label__="Person", name="Alice", age=30, email="alice@example.com")
G.add_node("p2", __label__="Person", name="Bob", age=45)
G.add_node("p3", __label__="Person", name="Charlie")  # missing 'age'

# Movies
G.add_node("m1", __label__="Movie", title="The Matrix", year=1999, rating=8.7)
G.add_node("m2", __label__="Movie", title="Inception", year=2010)

# Cities
G.add_node("c1", __label__="City", name="Los Angeles", country="USA")

# Relationships
G.add_edge("p1", "m1", __label__="ACTED_IN", role="Trinity")
G.add_edge("p2", "m1", __label__="ACTED_IN", role="Morpheus")
G.add_edge("p1", "m2", __label__="ACTED_IN", role="Ariadne")
G.add_edge("p2", "m2", __label__="DIRECTED")
G.add_edge("p1", "c1", __label__="LIVES_IN")
G.add_edge("p2", "c1", __label__="LIVES_IN")

print(f"Graph has {G.number_of_nodes()} nodes and {G.number_of_edges()} edges")

## Inspect the graph

`NetworkxInspector` scans the graph and produces a `GraphProfile` -- a frozen Pydantic model that summarizes everything about the graph's structure. No model definition is needed at this stage; the inspector reports what it finds.

In [ ]:
inspector = NetworkxInspector(G)
profile = inspector.inspect()

print("Profile source:   ", profile.source)
print("Profile timestamp:", profile.timestamp)
print("Node labels:      ", profile.node_labels)
print("Relationship types:", profile.relationship_types)

## Explore node type profiles

Each `NodeTypeProfile` contains the label, instance count, and a `PropertyProfile` for every property observed across all instances of that type. The property profile tracks completeness (what fraction of nodes have the property) and observed Python types.

In [ ]:
for label, ntp in profile.node_type_profiles.items():
    print(f"--- {label} ({ntp.count} instances) ---")
    for prop_name, pp in ntp.property_profiles.items():
        print(
            f"  {prop_name:12s}  "
            f"completeness={pp.completeness:.0%}  "
            f"({pp.present_count}/{pp.total_count})  "
            f"types={pp.observed_types}"
        )
    print()

Notice that:

- `age` on `Person` has only 67% completeness -- `Charlie` is missing it
- `email` on `Person` has only 33% completeness -- only `Alice` has it
- `rating` on `Movie` has 50% completeness -- only `The Matrix` has it

The profile captures this factually. Whether these are *problems* depends on the model definition -- `email` and `rating` are optional, so their low completeness is expected. But `age` is required, so that's a data quality issue.

## Explore relationship type profiles

Each `RelationshipTypeProfile` contains the relationship type, count, source/target labels, property profiles, and cardinality statistics (min/max/avg outgoing degree from source nodes).

In [ ]:
for rel_type, rtp in profile.rel_type_profiles.items():
    print(f"--- {rel_type} ({rtp.count} instances) ---")
    print(f"  source labels: {rtp.source_labels}")
    print(f"  target labels: {rtp.target_labels}")
    if rtp.cardinality_stats:
        cs = rtp.cardinality_stats
        print(
            f"  cardinality:   min={cs.min_degree}, max={cs.max_degree}, "
            f"avg={cs.avg_degree:.1f}, sample_size={cs.sample_size}"
        )
    for prop_name, pp in rtp.property_profiles.items():
        print(
            f"  {prop_name:12s}  "
            f"completeness={pp.completeness:.0%}  "
            f"types={pp.observed_types}"
        )
    print()

## Validate the profile against the model

`validate_profile` compares the `GraphProfile` against the `GraphDataModel` and reports:

- **Errors**: missing required types, type mismatches, invalid endpoints, cardinality violations
- **Warnings**: incomplete required properties, unexpected labels
- **Info**: unexpected properties not in the model

In [ ]:
result = validate_profile(profile, model)

print("is_valid:", result.is_valid)
print(f"Errors:   {len(result.errors)}")
print(f"Warnings: {len(result.warnings)}")
print()

for issue in result.issues:
    print(f"  [{issue.severity.value}] [{issue.code}] {issue.message}")
    if issue.context:
        print(f"    context: {issue.context}")

The validation should report:

- A **warning** that `age` on `Person` is only 67% complete (it's a required property in the model)
- The result is still `is_valid: True` because incomplete required properties generate warnings, not errors -- the type exists and data is mostly there, but quality is imperfect

## Convert the model schema to a NetworkX graph

The `schema_to_networkx` function converts the model definition itself -- not instance data -- into a NetworkX `MultiDiGraph`. Nodes in this graph represent node types; edges represent relationship types. This is useful for programmatic analysis of the schema structure (topology, path analysis, etc.).

In [ ]:
schema_graph = schema_to_networkx(model)

print("Schema graph nodes:")
for node, attrs in schema_graph.nodes(data=True):
    print(f"  {node}: uid_field={attrs['uid_field']}, properties={attrs['properties']}")

print()
print("Schema graph edges:")
for src, tgt, attrs in schema_graph.edges(data=True):
    print(f"  {src} --[{attrs['label']}]--> {tgt}")
    print(
        f"    source_cardinality={attrs['source_cardinality']}, target_cardinality={attrs['target_cardinality']}"
    )

## Serialise the profile

Since `GraphProfile` is a frozen Pydantic model, it can be serialised to JSON. This is useful for storing profiles as CI artifacts, comparing profiles across time, or feeding them into downstream tools.

In [ ]:
profile_json = profile.model_dump_json(indent=2)
print(profile_json[:500], "\n...")